TP4 - Parte I
Grupo 6: Acosta, Armelini y Freire.

En este Jupyter Notebook resolvemos la Parte I del TP4 de la materia. Análisis de la base de hogares y tipo de ocupación utilizando la EPH del 1T de 2005 y 2025.

Pregunta 1.1:
    Exploren el diseño de registro de la base de hogar: a priori, ¿qué variables creen
pueden ser predictivas de la desocupación y seria útil incluir para perfeccionar el
ejercicio del TP3? Mencionen estas variables y justifiquen su elección.


Respuesta: en el documento de texto.

Pregunta 1.2:
    "Descarguen la base de microdatos de la EPH correspondiente al primer trimestre de 2005 y 2025 en formato .dta y .xls, respectivamente. La base de hogares se llama Hogar_t105.dta y usu_hogar_T125.xls, respectivamente. Eliminen todas las observaciones que no corresponden al aglomerado con el que están trabajando1 y unan ambos trimestres en una sola base. Unan, a la base de la encuesta individual de cada año, la base de la encuesta de hogar. Asegúrese de estar usando las variables CODUSU y NRO_Hogar para el merge".

Respuesta: en el código debajo.

In [12]:
import pandas as pd
from pathlib import Path

# Diclaimer: el siguiente código para descargar y inicializar las bases requiere que las mismas esténd escargadas y guardadas en la carpeta "Downloads" de la computadora del usuario. Para su descarga, hemos subido las mismas al repositorio en esta misma carpeta del TP.

# Diclaimer 2: se precisa de la dependencia 'openpyxl' para leer archivos Excel, la cual se puede descargar con pip o conda.

# Se configura la ruta a bases de datos
repo_root = Path.cwd()               # asume que el kernel se lanzó desde la raíz del repo
data_dir = repo_root / "Datasets"    # subcarpeta 'Datasets' del TP4 del repo

# Se cargan las bases 2005 y 2025
ind_2005 = pd.read_stata(data_dir / "Individual_t105.dta")
hog_2005 = pd.read_stata(data_dir / "Hogar_t105.dta")
ind_2025 = pd.read_excel(data_dir / "usu_individual_T125.xlsx")
hog_2025 = pd.read_excel(data_dir / "usu_hogar_T125.xlsx")

hog_2005.info()
ind_2005.head()
ind_2025.head()

ind_2025 = ind_2025[ind_2025["AGLOMERADO"] == 3]
hog_2025 = hog_2025[hog_2025["AGLOMERADO"] == 3]

hog_2005.head()

ind_2005 = ind_2005[ind_2005["aglomerado"] == "Bahía Blanca - Cerri"]
hog_2005 = hog_2005[hog_2005["aglomerado"] == "Bahía Blanca - Cerri"]


## Pasamos todos los nombres a mayuscula

hog_2005.columns = hog_2005.columns.str.upper()
ind_2005.columns = ind_2005.columns.str.upper()

## Se une respectivamente a hogares e individuos en dos bases únicas vía concatenación.

hogares = pd.concat([hog_2025, hog_2005], axis=0, ignore_index=True)
individuos = pd.concat([ind_2025, ind_2005], axis=0, ignore_index=True)

## Merge entre individuos y hogares.
base_merged = individuos.merge(
    hogares,
    on=["CODUSU", "NRO_HOGAR", "ANO4"],
    how="left"
)

# Guardamos la base merged, solopor seguridad
base_merged.to_excel(data_dir / "Base_merged.xlsx", index=False)
print("Guardado en:", data_dir)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13597 entries, 0 to 13596
Data columns (total 88 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   CODUSU      13597 non-null  object  
 1   nro_hogar   13597 non-null  float64 
 2   realizada   13597 non-null  category
 3   ano4        13597 non-null  float64 
 4   trimestre   13597 non-null  category
 5   region      13597 non-null  category
 6   mas_500     13597 non-null  object  
 7   aglomerado  13597 non-null  category
 8   pondera     13597 non-null  float64 
 9   iv1         13597 non-null  category
 10  iv1_esp     13597 non-null  object  
 11  iv2         13597 non-null  category
 12  iv3         13597 non-null  category
 13  iv3_esp     13597 non-null  object  
 14  iv4         13597 non-null  category
 15  iv5         13597 non-null  category
 16  iv6         13597 non-null  category
 17  iv7         13597 non-null  category
 18  iv7_esp     13597 non-null  object  
 19  iv8 

Pregunta 1.3:
    "Limpien la base de datos tomando criterios que hagan sentido. Explicar cualquier
decisión comoeltratamientodevaloresfaltantes(missingvalues),extremos(outliers),
o variables categóricas. Justifique sus decisiones."

Respuesta: (código debajo, redacción en documento)

In [22]:
import numpy as np

## Iniciamos la limpieza y armonización de la base combinada.
base_def = base_merged.copy()

## Disclaimer: No se eliminan los NaN al menos inicialmente. Más detalle explicado en el documento.

# Se convierten a numéricas algunas columnas que pueden estar (mal)cargadas como texto (object/string).
vars_num = ['IX_TOT','IX_MEN10','IX_MAYEQ10','II1','II2','II3_1','II5_1','II6_1','IV2','CH06','PP03D','PP3E_TOT','PP3F_TOT','PP04B2','PP04B3_MES','PP04B3_ANO','PP04B3_DIA', 'PP05B2_MES','PP05B2_ANO','PP05B2_DIA','PP06K_SEM','PP06K_MES','PP06L','PP08G_DSEM','PP08G_DMES','PP08H','PP11B2_MES','PP11B2_ANO','PP11B2_DIA','PP11G_ANO','PP11G_MES','PP11G_DIA']
base_def[vars_num] = base_def[vars_num].apply(pd.to_numeric, errors='coerce') # (lo que no sea número se vuelve NaN).

# Se reemplazan valores negativos en columnas numéricas por NaN.
num_cols = base_def.select_dtypes(include="number").columns
base_def[num_cols] = base_def[num_cols].mask(base_def[num_cols] < 0) # .mask() "enmascara" (pone NaN) donde la condición es True.

# Se estandarizan algunas variables categóricas dicotómicas a valores numéricos. 
cols = ['V1','V2','V3','V4','V5','V5_01','V5_02','V5_03','V6','V7','V8','V9','V10','V11','V11_01','V11_02','V12','V13','V14','V15','V16','V17','V18','V19_A','V19_B','H15','II3','II5','II6','REALIZADA','CH13','CH09']

for c in cols:
    base_def[c] = (
        base_def[c]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            '1': 1,
            '2': 2,
            'si': 1,
            'sí': 1,
            'Si': 1,
            'Sí': 1,
            'no': 2,
            'No': 2
        })
    )
    # Convertir todo lo que quedó como texto a número/NaN
    base_def[c] = pd.to_numeric(base_def[c], errors='coerce')

# Se armonizan dos variables clave que serán utilizadas en el análisis: NIVEL_ED y ESTADO.
# ---
# Armonización de NIVEL_ED entre bases 2005 y 2025
print(base_def['NIVEL_ED'].unique()) # Vemos los valores únicos que toma la variable para saber cómo armonizar.
# Reemplazo de valores a su definición numérica.
base_def['NIVEL_ED'] = base_def['NIVEL_ED'].replace({
    'Primaria Incompleta (incluye educación especial)': 1,
    'Primaria Completa': 2,
    'Secundaria Incompleta': 3,
    'Secundaria Completa': 4,
    'Superior Universitaria Incompleta': 5,
    'Superior Universitaria Completa': 6,
    'Sin instrucción': 7,
    'Ns./Nr.': 9
})

# En la EPH, NIVEL_ED toma los valores 1 a 6 de manera incremental en el nivel educativo alcanzado, pero el valor 7 indica "sin instrucción". Por lo tanto, para mantener la lógica incremental, reasignamos el valor 0 a "sin instrucción". La variable NIVEL_ED computa con el valor 9 los casos de "no sabe" o "no responde", los cuales convertimos a NaN para que no afecten el cálculo de la nueva variable.

base_def["NIVEL_ED"] = base_def["NIVEL_ED"].replace({7: 0, 9: np.nan}) # 'Sin instrucción' -> 0 ; 'Ns/Nr' -> NaN.

print("Armonización de NIVEL_ED completada.")

# Armonización de ESTADO entre bases 2005 y 2025
print(base_def['ESTADO'].unique())
# Reemplazo de valores a su definición numérica.
base_def['ESTADO'] = base_def['ESTADO'].replace({
    'Ocupado': 1,
    'Desocupado': 2,
    'Inactivo': 3,
    'Menor de 10 años': 4
})

print("Armonización de ESTADO completada.")
# ---

# Guardamos la base final limpia
base_def.to_excel(data_dir / "Base_def.xlsx", index=False)
print("Guardado en:", data_dir)

C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\1877730397.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\1877730397.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({
C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\1877730397.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in 

[3 4 6 5 2 7 1 'Primaria Completa'
 'Primaria Incompleta (incluye educación especial)'
 'Superior Universitaria Completa' 'Secundaria Incompleta'
 'Superior Universitaria Incompleta' 'Sin instrucción'
 'Secundaria Completa']
Armonización de NIVEL_ED completada.
[1 3 4 2 'Ocupado' 'Inactivo' 'Menor de 10 años' 'Desocupado']
Armonización de ESTADO completada.
Guardado en: c:\Users\coding\bigdata\TPs_BigData2025\TP4\Datasets


Pregunta 1.4:
    "Construyan variables (mínimo 3) que no estén en la base pero que sean relevantes
para predecir individuos desocupados (por ejemplo, la proporción de personas que
trabajan en el hogar)."

Respuesta: (código debajo, redacción en documento)

In [23]:
## Se crean 3 variables adicionales:

# ============================
## 1) Máxima educación en el hogar (max_ed_h)
# ============================
    # -> Indica el nivel educativo más alto alcanzado por algún miembro del hogar. Da idea del clima educativo del hogar capturando el "techo" de capital humano y de posibilidad de "efecto par" (efecto impulso a los demás miembros), que puede influir en la probabilidad de desocupación. Más detalle en el documento.

# Cálculo de la variable max_ed_h.
base_def['max_ed_h'] = base_def.groupby(['CODUSU', 'NRO_HOGAR'])['NIVEL_ED'].transform('max')
# 'transform' calcula el max del grupo y lo pega en todas las filas de ese grupo.

print("1ra variable creada: 'max_ed_h'")


# ============================
## 2) Tasa de "dependencia" (tdep)
# ============================
    # -> Relación entre miembros dependientes (menores de 14 años y mayores de 65 años) y miembross en edad activa (14 a 65 años) en el hogar. Puede ser un predictor importante en cuanto a que requiere que la persona dedique más tiempo a quedarse en su casa, especialmente en caso de mujeres.  Más detalle en el documento.
    
# Se crean dummies auxiliares para dependientes (menores de 14 años y mayores de 65 años) y en edad activa (entre 14 y 65 años).
base_def['es_dependiente'] = np.where((base_def['CH06'] < 14) | (base_def['CH06'] > 65), 1, 0)
base_def['edad_activa'] = np.where((base_def['CH06'] >= 14) & (base_def['CH06'] <= 65), 1, 0)

# Se computan las sumas por hogar.
cant_dep = base_def.groupby(['CODUSU', 'NRO_HOGAR'])['es_dependiente'].transform('sum')
cant_edad_act = base_def.groupby(['CODUSU', 'NRO_HOGAR'])['edad_activa'].transform('sum')
# Usamos transform('sum') para tener el total del hogar repetido en cada fila

# Cálculo de la tasa de dependencia.
# 1. Calculamos primero una tasa de dependencia temporal con NaN para no romper nada
tasa_temp = cant_dep / cant_edad_act.replace(0, np.nan)
# 2. Reemplazamos los infinitos (NaNs por div/0) con el valor máximo de la muestra + un delta
max_val = tasa_temp.max() # Máximo de la muestra
delta = tasa_temp[tasa_temp > 0].min()  # delta = Mínimo positivo de la muestra
# Fallback por seguridad: si delta es NaN (ej. toda la muestra es 0), usar un default
if pd.isna(delta): 
    delta = 0.01 # Cappeo arbitrario auxiliar en caso de delta=NaN.

base_def['tdep'] = tasa_temp.fillna(max_val + delta)

# Estamos "cappeando" la tasa de dependencia en casos de completa dependencia asignando un límite superior razonable, ya que si un hogar no tiene miembros en edad activas (denominador 0) pero sí tiene dependientes, la tasa es teóricamente infinita. Económicamente, esto indica dependencia máxima. Por ende, de esta manera se mantiene el orden relativo entre hogares (estos hogares tienen la mayor dependencia).

# Establecemos ese delta lo "menos arbitrario" posible, utilizando el valor mínimo positivo de la tasa en la muestra. Con este criterio, se mantiene el orden relativo (ranking) poniendo a los de "infinita" dependencia en la cima utilizando una brecha "data-driven" con el "salto mínimo" de dependencia observado en la muestra.
    
# Vemos valores
print(f"Máximo finito: {max_val:.4f} | Delta (mínimo positivo): {delta:.4f}")
print(f"Valor imputado a casos sin activos: {max_val + delta:.4f}")
    
print("2da variable creada: 'tdep'")


# ============================
## 3) Hogar mantenido con un ingreso no laboral "adicional" (ADICIONAL)
# ============================
    # -> Dummy para hogares cuyos miembros se mantienen gracias a un ingreso no laboral (subsidio, ayudas, etc.), descartando pensiones, ahorro/interés, y renta. Puede ser un predictor importante de desocupación al reducir la necesidad de ingresos laborales, desalentando la búsqueda activa de empleo. Más detalle en el documento.

# Previamente, hacemos una armonización de V5 entre bases 2005 y 2025, para poder incluirla en la construcción de la variable ADICIONAL. Lo hacemos colapsando la desagregación presente en la base 2025 de las 3 subcategorías a una sola V5 como en la base 2005.
# ---
# Armonización de V5 entre bases 2005 y 2025
cols_v5_desag = ['V5_01', 'V5_02', 'V5_03']
cols_presentes = [c for c in cols_v5_desag if c in base_def.columns]

if cols_presentes:
    # Calculamos el valor para 2025 (colapsando las desagregadas)
    v5_combinada = base_def[cols_presentes].replace({1: 1, 2: 0}).fillna(-1).max(axis=1)
    v5_combinada = v5_combinada.replace(-1, np.nan) # Restauramos el -1 a NaN
    # Explicación:
        # 1=Sí, 2=No. Convertimos 1->1, 2->0 (y otros a 0).
        # Tomamos el MAX: Si alguna subcategoría es 1, V5 pasa a ser 1.
        # Tratamiento de NaNs: Como las filas del año 2005 no tienen datos en V5_1, V5_2 y V5_3, para ellas esos valores son NaN (vacíos), podríamos romper esos datos si intentáramos operar con esos NaNs. Por eso los rellenamos, temporalmente, con -1 para que no afecten el cálculo del MAX. Así, como las filas de 2005 ya tienen un valor en V5, no se ven afectadas (fillna ignora nuestro -1 calculado y respeta el dato original). Ahora, al momento de combinar V5, los datos NaN de la base 2025 en v5_combinada (es decir, los que no respondieron a ninguna de las subcategorías), en lugar de tratarlos como 0 los volvemos a transformar de -1 a NaN (elegimos -1 como "placeholder" o relleno temporal también para evitar confusiones con 0, que es haber respondido "No").
    
    # Rellenamos los huecos en la columna V5 original (los que corresponden a 2025)
    base_def['V5'] = base_def['V5'].fillna(v5_combinada)

# Limpieza posterior: Borramos las columnas desagregadas V5_01, etc., ya que la info está guardada en V5.
base_def.drop(columns=cols_presentes, inplace=True, errors='ignore')

print("Armonización V5 completada (respetando Missings reales).")
# ---

# Variables seleccionadas (excluyendo V1=Trabajo y V2=Jubilación/Pensión)
# V3: Indemnización Despido, V4: Seguro Desempleo, V5: Subsidio, V6-V7: Ayuda en especies, V12: Ayudas monetarias, V14: Préstamos de familiares/amigos, V17: Venta pertenencias, V18: Limosnas, juegos de azar, etc., V19_A: Trabajo niño, V19_B: Mendicidad niño.
ingresos_no_lab = ["V3", "V4", "V5", "V6","V7","V12","V14",'V17','V18','V19_A','V19_B']

# Explicar: Por qué no V1 (Trabajo), V2 (Jubilación/Pensión), V8 (Renta propiedad), V9 (Ganancias negocio), V10 (Intereses inversiones), V11 (Beca), V13 (Ahorros), V15 (Préstamos bancarios), y V16 (Cuotas tarjeta crédito) de las variables en la sección "Estrategias del hogar".
# V15 puede ser menos relevante porque sin ingresos laborales difícilmente se accede a un préstamo bancario...

# Generamos la dummy: 1 si el hogar recurrió a cualquiera de estas estrategias para subsistir, 0 si no (ninguna).
subset = base_def[ingresos_no_lab].replace({1: 1, 2: 0}) # Aseguramos 1/0 (principalmente para base 2005), los NaNs siguen siendo NaNs.
base_def["ADICIONAL"] = subset.max(axis=1)
# MAX para preservar NaNs: Si hay un 1 -> 1, si hay puros 0 y NaNs -> 0, y si todo es NaN -> NaN

print("3ra variable creada: 'ADICIONAL'")
print(base_def["ADICIONAL"].value_counts(dropna=False)) # Vemos conteo de 1, 0 y NaNs.

1ra variable creada: 'max_ed_h'
Máximo finito: 6.0000 | Delta (mínimo positivo): 0.1667
Valor imputado a casos sin activos: 6.1667
2da variable creada: 'tdep'
Armonización V5 completada (respetando Missings reales).
3ra variable creada: 'ADICIONAL'
ADICIONAL
0.0    1225
1.0     608
Name: count, dtype: int64


C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\2469685582.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_def['max_ed_h'] = base_def.groupby(['CODUSU', 'NRO_HOGAR'])['NIVEL_ED'].transform('max')
C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\2469685582.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_def['es_dependiente'] = np.where((base_def['CH06'] < 14) | (base_def['CH06'] > 65), 1, 0)
C:\Users\Agustin Acosta\AppData\Local\Temp\ipykernel_7904\2469685582.py:22: PerformanceWarning: DataFr

Pregunta 1.5:
    "Presenten estadísticas descriptivas de cinco variables de la encuesta de hogar que
ustedes creen que pueden ser relevantes para predecir desocupación. Comenten las
estadísticas obtenidas."

Respuesta: (código debajo, redacción en documento)

In [ ]:
## Se analiza estadística descriptiva de siguientes variables:
    
    ## i) Edad de los encuestados.
    
Base_def["CH06"].agg(
    media="mean",
    mediana="median",
    volatilidad="std",
    max="max",
    min="min"
)

    ## ii) Prob Niños en el hogar
    
Base_def["MENOR14HOG"].agg(
    media="mean",
    mediana="median",
    volatilidad="std",
    max="max",
    min="min"
)

    ## Histograma de edad.
    
    
    ## Desocupados por años de educación.

Pregunta 1.6:
    "En el TP3 calcularon la tasa de desocupación según INDEC y economía laboral,
para el 1er trimestre de 2025. Utilice una sola observación por hogar y sumen el
ponderador PONDERA que permite expandir la muestra de la EPH al total de la
población que representa ¿Cuál es la tasa de hogares con desocupación para su
aglomerado? ¿se asemeja dicha tasa a la reportada en el INDEC en sus informes?"

Respuesta: (código debajo, redacción en documento)